# Positional Encodings: Sinusoids, Rotations, and RoPE

Topic 9 read attention as a kernel smoother over a set, and a set has no order: one layer cannot tell *the dog bit the
man* from *the man bit the dog*. This notebook adds position, and reads every way of adding it as a **kernel over
offsets**. Sinusoids are a fixed stationary kernel that rules distance logarithmically; added to the content, they do
not make attention relative. RoPE rotates queries and keys so every score depends only on the offset, and up to the
learned projections it is the only linear way to do so. ALiBi multiplies Topic 9's content kernel by an exponential
kernel in time, which bounds its reach at every length.

Then we train single causal heads with each scheme on a task that needs order — *which cue came last?* — at 32 tokens,
and read them to 256, and on whole held-out filings: *which company was named last?*

Every number below is computed by `positional_encodings_geometry.py`, which asserts each claim. The cell below imports
it and trains the tournament's fifteen heads (about 15 s).

In [ ]:
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "positional_encodings_geometry.py").exists()
                       else here / "notebooks" / "positional-encodings-geometry"))
import positional_encodings_geometry as pe
from positional_encodings_geometry import HeadConfig

T = pe.tournament()
print("accuracy (mean of 3 seeds) at test length:", pe.TEST_LENGTHS)
for s in pe.SCHEMES:
    print(f"  {s:11}", "  ".join("  n/a" if np.isnan(np.mean(T['schemes'][s][n])) else f"{np.mean(T['schemes'][s][n]):.3f}"
                              for n in pe.TEST_LENGTHS))
for r in pe.RESCUES:
    print(f"  rope+{r:6}", "  ".join(f"{np.mean(T['rescues'][r][n]):.3f}" for n in pe.TEST_LENGTHS))

## 1. Order, and how a mask leaks it

Topic 9's Proposition 3 says bidirectional attention is permutation-equivariant, and one causal layer reads its prefix
as a multiset. A multiset has a size, though.

**Prop 1 (a causal layer counts but does not order).** (i) With equal scores, a BOS value $e$ at position $0$, and zero
values elsewhere, one causal layer outputs $o_t = e/(t+1)$, so $t$ is recovered exactly. (ii) The size of the prefix's
multiset is its position; its order is invisible.

So a causal stack without positions can write absolute position in layer 1 and use it in layer 2 (Haviv et al. 2022).
A single NoPE head cannot: on the cue task it is bounded by the best multiset reader, which guesses the most frequent cue
symbol.

In [ ]:
pe.test_causal_counter()
print("o_t for t = 0..5:", np.round(pe.causal_counter(5), 6))
pe.test_nope_reads_a_bag()
rng = np.random.default_rng(0)
for n in pe.TEST_LENGTHS:
    bayes, se = pe.bag_bayes_accuracy(n, pe.CUE_RATE, 20_000, rng)
    print(f"n = {n:3}: best multiset reader {bayes:.3f} ± {se:.3f}   NoPE heads {np.round(T['schemes']['nope'][n], 4)}")

## 2. Position as a set of clocks

Let $d$ be even, the planes $j = 0, \dots, d/2 - 1$, and the frequencies $\omega_j = b^{-2j/d}$ with base $b = 10^4$.
Write $R(\varphi)$ for the rotation by $\varphi$ in the plane and $R_\omega(k) = \bigoplus_j R(k\omega_j)$.

- **Def 1 (absolute additive encoding).** $h_m = x_m E + p(m)$, with a learned table $p$ (undefined past the training
  length) or a fixed code.
- **Def 2 (sinusoidal code; Vaswani et al. 2017).** $p(m) = (\cos m\omega_0, \sin m\omega_0, \dots,
  \cos m\omega_{d/2-1}, \sin m\omega_{d/2-1})$: the hands of $d/2$ clocks turning at rates $1, b^{-2/d}, \dots$

**Prop 2 (shift is a rotation).** $p(m + k) = R_\omega(k)\, p(m)$, and $k \mapsto R_\omega(k)$ is a homomorphism
$\mathbb Z \to SO(d)$. Every position lies on the sphere $\|p(m)\|^2 = d/2$, and plane $0$ alone makes the code
injective, since $\omega_0 = 1$ and $\pi$ is irrational.

In [ ]:
pe.test_shift_is_rotation()
w = pe.frequencies(16)
m, k = 1234, 77
print("‖p(m + k) − R(k)p(m)‖ =", np.linalg.norm(pe.sinusoidal([m + k], w)[0] - pe.rotation(k, w) @ pe.sinusoidal([m], w)[0]))
print("‖p(m)‖² =", round(float((pe.sinusoidal([m], w) ** 2).sum()), 12), "= d/2")

**Thm 1 (the stationary kernel, and the logarithmic ruler).** (i) $\langle p(m), p(n)\rangle = \kappa_d(n - m)$ with
$\kappa_d(\Delta) = \sum_j \cos(\Delta\omega_j)$; every Gram matrix is Toeplitz and positive semidefinite, and
$(2/d)\kappa_d(\Delta) = \mathbb E\cos(\Delta\Omega_d)$ is the characteristic function of $\Omega_d$, uniform on the
$d$ atoms $\pm\omega_j$. (ii) As $d \to \infty$, $\Omega_d$ converges in distribution to the symmetrized log-uniform law
on $[1/b, 1]$, so for every fixed $\Delta \ne 0$

$$\tfrac{2}{d}\kappa_d(\Delta) \to \frac{\operatorname{Ci}(|\Delta|) - \operatorname{Ci}(|\Delta|/b)}{\ln b}
\approx 1 - \frac{\gamma + \ln|\Delta|}{\ln b} \quad (1 \ll |\Delta| \ll b).$$

Similarity falls by $\ln 2/\ln b \approx 0.075$ per doubling of distance: a ruler that is logarithmic in distance.

In [ ]:
pe.test_stationary_kernel()
pe.test_logarithmic_ruler()
deltas = [1, 4, 16, 64, 256, 1024]
for d in (64, 512, 8192):
    print(f"d = {d:5}:", "  ".join(f"{x:.6f}" for x in pe.kernel(deltas, pe.frequencies(d)) / (d / 2)))
print("Ci limit:", "  ".join(f"{x:.6f}" for x in pe.kernel_limit(deltas)))
print("ruler:   ", "  ".join(f"{x:.6f}" for x in pe.log_ruler(deltas)))

**Thm 2 (which score matrices see only offsets).** For distinct $\omega_j \in (0, \pi)$, $p(m)^\top M p(n)$ depends
only on $n - m$ (on all of $\mathbb Z$, or on one window of $d + 1$ positions) iff $M$ commutes with $R_\omega(1)$, iff
$M = \bigoplus_j (a_j I + b_j J)$: a $d$-dimensional subspace of $\mathbb R^{d\times d}$. On it the score is the
trigonometric polynomial $\sum_j a_j\cos(\Delta\omega_j) + b_j\sin(\Delta\omega_j)$.

So adding a stationary code to the content does not make attention relative: the learned $M = W_Q W_K^\top$ must land in
the commutant, and content–position cross terms enter besides. The relativity defect
$\delta(M) = \|M - \Pi(M)\|_F / \|M\|_F$ has $\mathbb E\,\delta^2 = 1 - 1/d$ for a Gaussian $M$. Exactness is fragile:
over a short window the slow planes barely move, and the window's constraint on them is nearly void.

In [ ]:
pe.test_commutant()
pe.test_sinusoidal_head_defect()
print("trained sinusoidal heads' δ(W_Q W_Kᵀ):",
      [round(pe.relativity_defect(pe.train_head(HeadConfig("sinusoidal", s)).P["Wq"]
                                  @ pe.train_head(HeadConfig("sinusoidal", s)).P["Wk"].T), 6) for s in pe.TOUR_SEEDS],
      " random: √(1 − 1/16) =", round(np.sqrt(1 - 1 / 16), 6))
for n in (16, 32, 64, 256, 4096):
    print(f"condition number of the sinusoid matrix over {n:4} positions (d = 16): {pe.window_condition(n, pe.TOUR_FREQS):.3e}")

## 3. Relative position: biases and the ALiBi kernel

- **Def 3 (relative schemes).** Shaw, Uszkoreit, and Vaswani (2018) add a learned vector per clipped offset to the key;
  T5 (Raffel et al. 2020) adds a learned scalar per log-spaced offset bucket; ALiBi (Press, Smith, and Lewis 2022)
  subtracts $s_h(t - j)$ with fixed slopes $s_h = 2^{-8h/H}$. Our head uses $s = 1/2$, ALiBi's first slope at $H = 8$.

**Thm 3 (ALiBi is a product kernel).** Let the content scores lie in an interval of width $W$. (i) The weights are
$\alpha_{tj} \propto e^{c_{tj}}\, e^{-s(t - j)}$: Topic 9's Nadaraya–Watson smoother with a content × Matérn-½ product
kernel. (ii) The weight on offsets $\ge D$ is at most $e^{W} e^{-sD}/(1 - e^{-s})$ at every length. (iii) A key at
offset $r$ weighs at most $e^{W - sr}$ times the offset-$0$ key: past the horizon $W/s$ it weighs less, whatever its
content. ALiBi extrapolates in length *because* it cannot look far.

In [ ]:
pe.test_t5_buckets()
starts = {}
for r in range(200):
    starts.setdefault(pe.t5_bucket(r), r)
print("T5 bucket starts past 16:", {b: r for b, r in starts.items() if b >= 16})
pe.test_alibi_product_kernel()
print("Thm 3 (ii) with W = 20, s = 1/2: weight past D =", {D: f"{pe.alibi_tail_bound(20, 0.5, D):.3g}" for D in (40, 60, 80)})

## 4. Rotations: RoPE

- **Def 4 (RoPE; Su et al. 2024).** $\tilde q_m = R_\theta(m)\, q_m$ and $\tilde k_n = R_\theta(n)\, k_n$ with
  $\theta_j = b^{-2j/d}$: Def 2's frequencies, applied multiplicatively.

**Prop 3 (relative by construction, and isometric).** $\langle R_\theta(m) q, R_\theta(n) k\rangle =
\langle q, R_\theta(n - m) k\rangle$ and $\|R_\theta(m)q\| = \|q\|$: in Topic 9's factorization the design weights
$e^{\|k\|^2/2\tau}$ carry no position, and RoPE moves only angles. For fixed $q, k$ the offset score
$s(\Delta) = \sum_j \rho_j\cos(\Delta\theta_j + \varphi_j)$ is a trigonometric polynomial, with plane amplitudes
$\rho_j = \|q_j\|\|k_j\|$.

In [ ]:
pe.test_rope_relative_isometric()
rng = np.random.default_rng(1)
q, k = rng.normal(size=16), rng.normal(size=16)
for m in (0, 100, 5000):
    print(f"m = {m:4}, n = m + 7:", round(float(pe.rotate(q[None], [m], pe.TOUR_FREQS)[0] @ pe.rotate(k[None], [m + 7], pe.TOUR_FREQS)[0]), 10))

**Thm 4 (RoPE is the general linear relative encoding).** Let $L_m \in \mathbb R^{d\times d}$ with $L_0 = I$. Then
$\langle L_m q, L_n k\rangle$ depends only on $(q, k, n - m)$ for all $q, k, m, n$ iff $L_m = U^m$ for an orthogonal $U$,
iff $L_m = Q\,(\bigoplus_i R(m\theta_i) \oplus I_p \oplus (-1)^m I_r)\,Q^\top$. The proof of the first step is three
lines: $m = n$ makes each $L_m$ orthogonal, $G(k) = L_k$, and $L_{m+k} = L_m L_k$. The second is the real normal form of
an orthogonal matrix (Horn and Johnson 2013). $Q$ is absorbed by $W_Q$ and $W_K$, so up to the learned projections every
linear relative encoding is RoPE with some frequencies, plus unrotated and Nyquist planes.

In [ ]:
pe.test_general_relative_encoding()
Z, _ = np.linalg.qr(np.random.default_rng(2).normal(size=(7, 7)))
Q, angles, p, r = pe.real_normal_form(Z)
print("a random orthogonal 7 × 7: angles", np.round(angles, 6), "+1 axes", p, "−1 axes", r)
print("max |U^5 − normal-form power| =", np.abs(pe.orthogonal_family(Z, [5])[0] - pe.normal_form_power(Q, angles, p, r, 5)).max())

**Prop 4 (what "long-term decay" means).** (i) Su et al.'s bound: with $h_j = q_j\bar k_j$ and
$S_j = \sum_{l<j} e^{i\Delta\theta_l}$, summation by parts gives
$|\sum_j h_j e^{i\Delta\theta_j}| \le (\max_j |h_{j+1} - h_j|) \sum_j |S_j|$, and the envelope $(2/d)\sum_j|S_j|$ falls
with $\Delta$ on average. (ii) A matched pair with isotropic planes has $\mathbb E\langle q, R(\Delta) q\rangle =
\sigma^2\kappa_d(\Delta)$: Thm 1's ruler. (iii) For independent $q, k$ the law of $\langle q, R(\Delta) k\rangle$ does
not depend on $\Delta$. The decay is a property of an envelope and of matched pairs, not a guarantee.

In [ ]:
pe.test_long_term_decay()
w = pe.frequencies(128)
env = np.array([pe.abel_envelope(np.ones(64), dl, w)["envelope"] for dl in range(1, 4096)])
print("envelope, mean over each octave [2^k, 2^(k+1)):", np.round([env[2**k - 1:2 ** (k + 1) - 1].mean() for k in range(12)], 3))

## 5. Past the training window

The cue task: $8$ symbols, each token a cue with probability $0.1$, a query at the end; the target is the symbol on the
most recent cue, so the head needs content (*is this a cue?*) and order (*which came last?*). One causal head,
$d = 16$, trained for 1,500 Adam steps at length $32$ with hand-written gradients, three seeds per scheme, and $2{,}000$
test sequences per length shared by every head.

In [ ]:
pe.test_gradients()
pe.test_tournament()
for s in pe.SCHEMES:
    for n in pe.TEST_LENGTHS:
        v = T["schemes"][s][n]
        if not np.isnan(v[0]):
            print(f"{s:11} n = {n:3}: mean {np.mean(v):.6f}  range {min(v):.4f}–{max(v):.4f}")

Training sees offsets $0$ to $31$ only. The RoPE head's score for a cue at offset $r$ is Prop 3's trigonometric
polynomial: it falls over the window, as the task rewards, and then does whatever its planes do. Only planes $0$ and
$1$ turn a whole circle in $32$ positions; the head puts its recency in planes $2$ ($\lambda = 62.8$) and $3$
($\lambda = 198.7$), whose half-turns fit the window, and they turn back.

In [ ]:
pe.test_rope_rebound()
for s in pe.TOUR_SEEDS:
    P = pe.train_head(HeadConfig("rope", s)).P
    sc = pe.learned_offset_score(P, "rope", np.arange(256))
    rho, _ = pe.trig_form(*pe.cue_query_key(P))
    trough = 1 + int(np.argmax(np.diff(sc[1:]) > 0))
    peak = 32 + int(np.argmax(sc[32:]))
    print(f"seed {s}: s(1) = {sc[1]:.3f}, s(16) = {sc[16]:.3f}, trough at {trough} ({sc[trough]:.3f}), "
          f"rebound peak at {peak} ({sc[peak]:.3f});  ρ_j/τ = {np.round(rho / 4, 2)}")

- **Def 5 (context-extension maps)**, for target length $L' = \kappa L$: *PI* (Chen et al. 2023) maps positions
  $m \mapsto m/\kappa$; *NTK-aware* (bloc97 2023) maps $b \mapsto b\kappa^{d/(d-2)}$; *YaRN* (Peng et al. 2024) ramps
  each plane between the two by $r_j = L/\lambda_j$ and scales the logits by $(0.1\ln\kappa + 1)^2$.

**Thm 5 (wavelength census and rescalings).** (i) Plane $j$ turns a whole circle over training offsets iff
$(L - 1)\theta_j \ge 2\pi$; the rest see angles past $L$ never seen in training. (ii) PI keeps every test offset within
one position of the trained range, at a $\kappa$-fold loss of resolution. (iii) NTK-aware leaves plane $0$ alone and
gives the slowest plane exactly $1/\kappa$. (iv) Linear interpolation of $a(\Delta) = \operatorname{Re}\sum_j h_j
e^{i\Delta\theta_j}$ between neighbors errs by at most $\tfrac12(\Delta - \Delta_1)(\Delta_2 - \Delta)\sum_j |h_j|\theta_j^2$,
and $\sum_j\theta_j^2 = (1 - b^{-2})/(1 - b^{-4/d})$. Chen et al.'s Theorem 2.1 states the constant $d/(4\ln b)$, which
their eq. (14) obtains from $c^x \le 1 + x\ln c$; convexity gives the reverse, and the stated bound fails at small $d$.

In [ ]:
pe.test_wavelength_census()
pe.test_interpolation_bound()
c = pe.wavelength_census(32, 16)
print("wavelengths:", ", ".join(f"{x:.1f}" for x in c["wavelengths"]), " complete turns:", c["n_complete"])
for d in (16, 64, 128, 512):
    w, h = pe.frequencies(d), np.ones(d // 2)
    err = abs(np.cos(w / 2).sum() - (d // 2 + np.cos(w).sum()) / 2)
    print(f"d = {d:3}: error at ½ {err:.6f}   stated bound {pe.chen_stated_bound(h, d, 0, 1, 0.5):.6f}   "
          f"sharp bound {pe.interpolation_bound(h, w, 0, 1, 0.5):.6f}")
for r in pe.RESCUES:
    print(f"rope+{r:4}", "  ".join(f"n = {n}: {np.mean(T['rescues'][r][n]):.6f}" for n in pe.TEST_LENGTHS[1:]))

## 6. Finance case study: which company was named last?

A filing names a parent and its subsidiaries and then refers back to "the company". At every position of a held-out
synthetic filing we ask which company was named most recently. Each entity mention collapses to one token; an ORG
mention is a cue labeled by its company's order of first appearance. The heads are retrained on 32-token windows of the
training filings, so the only shift at test time is length: whole filings of median length 198, where 41% of the
queries have their answer more than 32 tokens back. EDGAR's excerpts carry no entity spans, so they sit this one out.

In [ ]:
pe.test_filing_reader_matches_forward()
pe.test_filings()
fb = pe.filing_benchmark()
print(f"{fb['n_queries']} queries; by gap band {[f'{a}–{b}' if b < 10**9 else f'{a}+' for a, b in pe.GAP_BANDS]}: {fb['band_counts']}")
for name, rows in fb["variants"].items():
    bands = np.mean([r["bands"] for r in rows], axis=0)
    print(f"{name:10} all {np.mean([r['all'] for r in rows]):.4f}   by band {np.round(bands, 3)}")

In [ ]:
for a in pe.alibi_forecast():
    print(f"ALiBi trained on {a['source']:9} seed {a['seed']}: W = {a['W']:.3f}, horizon W/s = {a['horizon']:.1f};  "
          f"inside half of it {a['within']:.3f} (n = {a['n_within']}), beyond it {a['beyond']:.3f} (n = {a['n_beyond']})")

## The lab's numbers, and the full harness

`viz_payload()` holds every number the lab shows: the clocks' kernel tables and the logarithmic ruler; the trained
sinusoidal head's drifting score against the commutant's constant one; the tournament's accuracies and every scheme's
learned offset score; the census with each rescue's per-plane scales; and one held-out filing with every reader's
answers.

In [ ]:
pe.test_viz_payload()
P = pe.viz_payload()
print("lab filing: held-out document", P["filing"]["index"], "with", len(P["filing"]["tokens"]), "tokens")
print("relativity defect shown:", P["relativity"]["defect"], "  ALiBi horizon shown:", P["filing"]["horizon"])

In [ ]:
assert pe._run_harness() == 0